# StreamForge — FLUX.1-dev in Pure Rust on Kaggle

Full BF16 FLUX.1-dev inference pipeline written in **pure Rust** using [candle](https://github.com/huggingface/candle).  
Streams the 24GB transformer **block-by-block** from disk through CPU RAM to GPU — peak VRAM ~4GB.

**Source code:** https://github.com/madtunebk/GPULoad

## What this notebook does
1. Mounts the pre-converted model dataset (flux_candle + vae_candle safetensors)
2. Downloads CLIP-L + T5-XXL weights from HuggingFace (~10GB, ~2 min)
3. Uses pre-built binaries OR builds from source if glibc mismatch
4. Runs the full pipeline: text_encoder → flux_gpu → vae_decode → PNG

## Required Kaggle Dataset
Add your dataset (containing `flux_candle.safetensors` and `vae_candle.safetensors`) via:  
**Add Data → Models → flux-dev-rust-candle** (by valicornea84)

It will be mounted at `/kaggle/input/models/valicornea84/flux-dev-rust-candle/other/default/1/`

## GPU
Enable GPU accelerator: **Settings → Accelerator → GPU T4 x2** (or P100)

In [ ]:
# Verify GPU is available
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], 
                        capture_output=True, text=True)
print(result.stdout)

import os
print(f"HOME={os.environ.get('HOME')}")
print(f"CUDA available:", os.path.exists('/usr/local/cuda'))

## Step 1 — Check model dataset is mounted

## Step 1b — HuggingFace Authentication

FLUX.1-dev is a **gated model** — you must:
1. Accept the license at https://huggingface.co/black-forest-labs/FLUX.1-dev
2. Create a **classic** Read token at https://huggingface.co/settings/tokens  
   ⚠️ Fine-grained tokens block gated repos by default — use classic OR enable "Access to public gated repositories" in the fine-grained token settings
3. Add it as a Kaggle Secret: **Add-ons → Secrets → Add** with name `HF_TOKEN`

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token  # older HF versions
print("HF token loaded")

In [ ]:
import os

DATASET_DIR = "/kaggle/input/models/valicornea84/flux-dev-rust-candle/other/default/1"
FLUX_MODEL  = f"{DATASET_DIR}/flux_candle.safetensors"
VAE_MODEL   = f"{DATASET_DIR}/vae_candle.safetensors"

for path in [FLUX_MODEL, VAE_MODEL]:
    size = os.path.getsize(path) / 1e9
    print(f"OK  {path}  ({size:.1f} GB)")

## Step 2 — Download CLIP-L + T5-XXL from HuggingFace

These weights stay in the HF cache at `~/.cache/huggingface/` which matches the hardcoded paths in the Rust binaries.

> Takes ~2 minutes on Kaggle's network.

In [ ]:
!pip install -q huggingface_hub

from huggingface_hub import snapshot_download
import os

HF_SNAPSHOT_DIR = snapshot_download(
    repo_id="black-forest-labs/FLUX.1-dev",
    allow_patterns=["text_encoder/**", "text_encoder_2/**", "tokenizer/**", "tokenizer_2/**"],
    token=os.environ["HF_TOKEN"],
    ignore_patterns=["*.bin"],
    resume_download=True,
    local_dir_use_symlinks=False,
 )
HF_REPO_DIR = os.path.dirname(os.path.dirname(HF_SNAPSHOT_DIR))
os.environ["STREAMFORGE_HF_REPO_DIR"] = HF_REPO_DIR

print("HF snapshot:", HF_SNAPSHOT_DIR)
print("HF repo dir:", HF_REPO_DIR)
print("Done.")

## Step 3 — Set up workspace, clone repo, and wire model paths

In [ ]:
import os, subprocess

WORKDIR = "/kaggle/working/streamforge"
MODEL_DIR = f"{WORKDIR}/models"
TEMP_DIR = f"{WORKDIR}/temp"
LORA_DIR = f"{WORKDIR}/loras"
BIN_DST = f"{WORKDIR}/bin"

for path in [MODEL_DIR, f"{MODEL_DIR}/tokenizer", TEMP_DIR, LORA_DIR, BIN_DST]:
    os.makedirs(path, exist_ok=True)

repo_dir = "/kaggle/working/GPULOAD"
if not os.path.exists(repo_dir):
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/madtunebk/GPULoad.git", repo_dir
    ], check=True)
else:
    subprocess.run(["git", "-C", repo_dir, "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", repo_dir, "reset", "--hard", "origin/main"], check=True)
print(f"repo    {repo_dir}")

for fname, src in [("flux_candle.safetensors", FLUX_MODEL), ("vae_candle.safetensors", VAE_MODEL)]:
    dst = f"{MODEL_DIR}/{fname}"
    if os.path.islink(dst) or os.path.exists(dst):
        os.remove(dst)
    os.symlink(src, dst)
    print(f"linked  {dst}")

tokenizer_candidates = [
    os.path.join(repo_dir, "models", "tokenizer", "tokenizer.json"),
    os.path.join(HF_SNAPSHOT_DIR, "tokenizer", "tokenizer.json"),
]
clip_tokenizer_src = next((path for path in tokenizer_candidates if os.path.exists(path)), None)
if clip_tokenizer_src is None:
    raise FileNotFoundError("Could not find tokenizer.json in repo or HF snapshot")

clip_tokenizer_dst = f"{MODEL_DIR}/tokenizer/tokenizer.json"
if os.path.islink(clip_tokenizer_dst) or os.path.exists(clip_tokenizer_dst):
    os.remove(clip_tokenizer_dst)
os.symlink(clip_tokenizer_src, clip_tokenizer_dst)
print(f"linked  {clip_tokenizer_dst} -> {clip_tokenizer_src}")

os.environ["STREAMFORGE_MODEL_DIR"] = MODEL_DIR
os.environ["STREAMFORGE_HF_REPO_DIR"] = HF_REPO_DIR
print(f"STREAMFORGE_MODEL_DIR={MODEL_DIR}")
print(f"STREAMFORGE_HF_REPO_DIR={HF_REPO_DIR}")

## Step 4 — Build from source on Kaggle (native, sm_75 for T4)

Installs Rust and compiles the binaries directly on Kaggle with CUDA enabled.

This notebook no longer depends on repo-shipped `dist/` binaries or a Candle patch step.

In [ ]:
import glob, os, shutil, time

if os.path.exists("/content/GPULOAD"):
    repo_dir = "/content/GPULOAD"
elif os.path.exists("/kaggle/working/GPULOAD"):
    repo_dir = "/kaggle/working/GPULOAD"
else:
    raise RuntimeError("Cannot find GPULOAD repo — clone it first")

BIN_DST = f"{WORKDIR}/bin"
print(f"repo_dir: {repo_dir}")

CUDA_ARCH = "75"

!curl https://sh.rustup.rs -sSf | sh -s -- -y --default-toolchain stable --profile minimal

found_cuda_dir = "/usr/local/cuda/lib64/stubs"
for d in ["/usr/local/cuda/compat", "/usr/lib/x86_64-linux-gnu", "/usr/lib/x86_64-linux-gnu/nvidia"]:
    hits = glob.glob(f"{d}/libcuda.so*")
    if hits:
        found_cuda_dir = d
        stub = f"{d}/libcuda.so"
        if not os.path.exists(stub):
            try:
                os.symlink(hits[0], stub)
            except PermissionError:
                os.makedirs("/tmp/cuda_stubs", exist_ok=True)
                tgt = "/tmp/cuda_stubs/libcuda.so"
                if not os.path.exists(tgt):
                    os.symlink(hits[0], tgt)
                found_cuda_dir = "/tmp/cuda_stubs"
        print(f"libcuda dir: {found_cuda_dir}")
        break

cargo_bin = os.path.expanduser("~/.cargo/bin")
os.environ["PATH"] = f"{cargo_bin}:{os.environ['PATH']}"
os.environ["RUSTFLAGS"] = (
    f"-C link-arg=-L{found_cuda_dir} "
    f"-C link-arg=-L/usr/local/cuda/lib64/stubs "
    f"-C link-arg=-L/usr/local/cuda/lib64"
 )
os.environ["CUDA_COMPUTE_CAP"] = CUDA_ARCH
os.environ["CUDA_PATH"] = "/usr/local/cuda"
os.environ["CUDA_ROOT"] = "/usr/local/cuda"
print(f"Building for sm_{CUDA_ARCH}")

t0 = time.time()
!cd {repo_dir} && \
    CUDA_COMPUTE_CAP={CUDA_ARCH} CUDA_PATH=/usr/local/cuda CUDA_ROOT=/usr/local/cuda \
    RUSTFLAGS="-C link-arg=-L{found_cuda_dir} -C link-arg=-L/usr/local/cuda/lib64/stubs -C link-arg=-L/usr/local/cuda/lib64" \
    STREAMFORGE_MODEL_DIR={MODEL_DIR} STREAMFORGE_HF_REPO_DIR={HF_REPO_DIR} \
    ~/.cargo/bin/cargo build --release --features cuda
print(f"Build done in {time.time()-t0:.0f}s  (sm_{CUDA_ARCH})")

for b in ["text_encoder", "flux_gpu", "vae_decode"]:
    shutil.copy2(f"{repo_dir}/target/release/{b}", f"{BIN_DST}/{b}")
    os.chmod(f"{BIN_DST}/{b}", 0o755)
    print(f"copied  {BIN_DST}/{b}")

!{BIN_DST}/flux_gpu --help | head -1

## Step 5 — Run the pipeline

Edit `PROMPT`, `WIDTH`, `HEIGHT`, `STEPS` below.

In [ ]:
import os

checks = [
    MODEL_DIR,
    f"{MODEL_DIR}/flux_candle.safetensors",
    f"{MODEL_DIR}/vae_candle.safetensors",
    f"{MODEL_DIR}/tokenizer/tokenizer.json",
    TEMP_DIR,
    BIN_DST,
    f"{BIN_DST}/text_encoder",
    f"{BIN_DST}/flux_gpu",
    f"{BIN_DST}/vae_decode",
    HF_REPO_DIR,
    os.path.join(HF_REPO_DIR, "snapshots"),
]

all_ok = True
for p in checks:
    exists = os.path.exists(p)
    status = "OK " if exists else "MISSING"
    if not exists:
        all_ok = False
    print(f"[{status}] {p}")

if not all_ok:
    raise RuntimeError("Pre-flight check failed — fix missing paths above before running pipeline")

print(f"\nSTREAMFORGE_MODEL_DIR={MODEL_DIR}")
print(f"STREAMFORGE_HF_REPO_DIR={HF_REPO_DIR}")
print("All checks passed.")

In [ ]:
PROMPT = "an ivory unicorn with a molten-gold horn and a pearl-white mane, standing knee-deep in moonlit water, liquid starlight dripping from its mane, a halo of fireflies and blue will-o-wisps swirling around it, gilded filigree armor on its chest, cathedral-tall ancient forest in the background, cinematic volumetric light, luminous mist, high fantasy oil painting, lavish detail, regal aura, masterpiece"
WIDTH = 768
HEIGHT = 1024
STEPS = 20
GUIDANCE = 3.5
SEED = 42
DTYPE = "f16"
LORA = None
LORA_SCALE = 1.0

In [ ]:
import os

PROMPT_EMBEDS = f"{TEMP_DIR}/prompt_embeds.safetensors"
LATENTS_PATH = f"{TEMP_DIR}/latents.safetensors"
OUTPUT_PATH = f"{TEMP_DIR}/output_rust.png"

os.environ["PATH"] = f"{BIN_DST}:{os.environ['PATH']}"
os.environ["STREAMFORGE_MODEL_DIR"] = MODEL_DIR
os.environ["STREAMFORGE_HF_REPO_DIR"] = HF_REPO_DIR
os.environ["STREAMFORGE_DTYPE"] = DTYPE

for stale_path in [PROMPT_EMBEDS, LATENTS_PATH, OUTPUT_PATH]:
    if os.path.exists(stale_path):
        os.remove(stale_path)
        print(f"removed stale file: {stale_path}")

print("Environment ready")
print(f"STREAMFORGE_MODEL_DIR={MODEL_DIR}")
print(f"STREAMFORGE_HF_REPO_DIR={HF_REPO_DIR}")
print(f"STREAMFORGE_DTYPE={DTYPE}")
print(f"PROMPT_EMBEDS={PROMPT_EMBEDS}")
print(f"LATENTS_PATH={LATENTS_PATH}")
print(f"OUTPUT_PATH={OUTPUT_PATH}")

In [ ]:
import shlex

te_cmd = [
    f"{BIN_DST}/text_encoder",
    PROMPT,
    "--device", "cuda",
]
if LORA:
    te_cmd += ["--lora", LORA, "--lora-scale", str(LORA_SCALE)]

te_cmd_str = shlex.join(te_cmd)
print("$", te_cmd_str)
get_ipython().system(te_cmd_str)

print("prompt embeds exists:", os.path.exists(PROMPT_EMBEDS))

In [ ]:
import shlex

fg_cmd = [
    f"{BIN_DST}/flux_gpu",
    "--embeddings", PROMPT_EMBEDS,
    "--out-latents", LATENTS_PATH,
    "--width", str(WIDTH),
    "--height", str(HEIGHT),
    "--steps", str(STEPS),
    "--guidance", str(GUIDANCE),
    "--seed", str(SEED),
]
if LORA:
    fg_cmd += ["--lora", LORA, "--lora-scale", str(LORA_SCALE)]

fg_cmd_str = shlex.join(fg_cmd)
print("$", fg_cmd_str)
get_ipython().system(fg_cmd_str)

print("latents exist:", os.path.exists(LATENTS_PATH))

In [ ]:
import shlex

vd_cmd = [
    f"{BIN_DST}/vae_decode",
    "--latents", LATENTS_PATH,
    "--out", OUTPUT_PATH,
]

vd_cmd_str = shlex.join(vd_cmd)
print("$", vd_cmd_str)
get_ipython().system(vd_cmd_str)

print("output exists:", os.path.exists(OUTPUT_PATH))

## Result

In [ ]:
from IPython.display import Image, display
import shutil, os

output_src = f"{WORKDIR}/temp/output_rust.png"
output_dst = "/kaggle/working/output_rust.png"
shutil.copy2(output_src, output_dst)

display(Image(filename=output_dst))
print(f"Saved to {output_dst}")